In [28]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [29]:
CSV_PATH = "IoT_Intrusion.csv"
MODEL_OUT = "iot_scanner_model.pkl"
RANDOM_STATE = 42
TEST_SIZE = 0.20

In [30]:
COLUMN_LIST = [
"flow_duration","Header_Length","Protocol Type","Duration","Rate","Srate","Drate",
"fin_flag_number","syn_flag_number","rst_flag_number","psh_flag_number","ack_flag_number",
"ece_flag_number","cwr_flag_number","ack_count","syn_count","fin_count","urg_count","rst_count",
"HTTP","HTTPS","DNS","Telnet","SMTP","SSH","IRC","TCP","UDP","DHCP","ARP","ICMP","IPv","LLC",
"Tot sum","Min","Max","AVG","Std","Tot size","IAT","Number","Magnitue","Radius","Covariance","Variance","Weight","label"
]

In [31]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at {CSV_PATH}. Put your CSV file there or change CSV_PATH.")

df = pd.read_csv(CSV_PATH)
print("Loaded dataset:", df.shape)

Loaded dataset: (1048575, 47)


In [32]:
missing = [c for c in COLUMN_LIST if c not in df.columns]
if missing:
    print("Warning: These expected columns are missing from the CSV:", missing)


use_cols = [c for c in COLUMN_LIST if c in df.columns]
print("Using columns:", len(use_cols))

Using columns: 47


In [33]:
df = df[use_cols].copy()
df = df.fillna(0)

if 'Protocol Type' in df.columns:
    if df['Protocol Type'].dtype == object:
        le_proto = LabelEncoder()
        df['Protocol Type'] = le_proto.fit_transform(df['Protocol Type'].astype(str))


categorical_candidates = ['Header_Length']

for c in df.columns:
    if df[c].dtype == object and c != 'label':
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))

if 'label' not in df.columns:
    raise KeyError("Target column 'label' not found in CSV.")
X = df.drop(columns=['label'])
y = df['label']

if y.dtype == object:
    le_y = LabelEncoder()
    y = le_y.fit_transform(y.astype(str))

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", pd.Series(y).value_counts())

Feature matrix shape: (1048575, 46)
Target distribution:
 6     161281
14    121205
13    101293
8      92395
10     91644
9      90823
12     80680
21     74787
20     59807
19     45207
1      24476
23     22115
25     20166
24     16952
7      10223
22      7019
4       6431
15      6431
16      4034
26      3007
27      2225
29      1863
18      1680
32       809
5        626
11       493
17       324
2        140
30       122
3        105
0         76
33        72
28        41
31        23
Name: count, dtype: int64


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

In [35]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [36]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print(report)
print("Confusion matrix:\n", cm)

out_dir="Training_output"
os.makedirs(out_dir, exist_ok=True)
pd.DataFrame(cm).to_csv(os.path.join(out_dir, "confusion_matrix.csv"), index=False)
with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
    f.write(report)

pd.DataFrame({"accuracy": [acc]}).to_csv(os.path.join(out_dir, "accuracy.csv"), index=False)


Accuracy: 0.9921369477624395
              precision    recall  f1-score   support

           0       1.00      0.07      0.12        15
           1       0.82      0.99      0.90      4895
           2       1.00      0.07      0.13        28
           3       1.00      0.43      0.60        21
           4       1.00      0.99      0.99      1286
           5       0.99      0.96      0.98       125
           6       1.00      1.00      1.00     32256
           7       0.99      1.00      0.99      2045
           8       1.00      1.00      1.00     18479
           9       1.00      1.00      1.00     18165
          10       1.00      1.00      1.00     18329
          11       0.96      0.95      0.95        99
          12       1.00      1.00      1.00     16136
          13       1.00      1.00      1.00     20259
          14       1.00      1.00      1.00     24241
          15       1.00      0.99      0.99      1286
          16       0.87      0.67      0.76       80

In [37]:
joblib.dump(model, MODEL_OUT)
print(f"Trained model saved to: {MODEL_OUT}")

Trained model saved to: iot_scanner_model.pkl


In [38]:
if hasattr(model,"feature_importances_"):
    fi=pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
    fi.to_csv(os.path.join(out_dir, "feature_importances.csv"), index=False)
    print("Top features:\n", fi.head(10))

Top features:
           feature  importance
39            IAT    0.192969
1   Header_Length    0.055144
2   Protocol Type    0.045840
38       Tot size    0.044197
18      rst_count    0.042101
0   flow_duration    0.041077
41       Magnitue    0.038357
35            Max    0.037405
36            AVG    0.036405
15      syn_count    0.035370
